In [ ]:
# 1D particle-in-a-box (finite-difference / tight-binding form)
# Plot (1) Eigen-energies (numeric + analytic) and (2) Wavefunction for a chosen state
#
# Notes:
# - Energies in eV
# - Dirichlet boundary conditions (psi=0 at ghost sites)
# - One chart per figure (per instructions)

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Parameters (feel free to edit)
# -------------------------
hbar = 1.055e-34           # J*s
m    = 9.10938356e-31      # kg (electron mass)
eps0 = 8.854e-12           # F/m (unused, kept for completeness)
q    = 1.602176634e-19     # C  (J -> eV factor)

Np   = 100                 # number of interior lattice points
a    = 1e-10               # grid spacing (m)
state_index = 25           # which eigenstate to visualize (1-based)

# -------------------------
# Discretization / Hamiltonian
# -------------------------
# Finite-difference Laplacian -> Tridiagonal kinetic energy operator
# t0 = hbar^2 / (2 m a^2) (in eV)
t0 = (hbar**2) / (2.0 * m * a**2) / q
L  = a * (Np + 1)  # "box" length due to Dirichlet at i=0 and i=Np+1

# Tridiagonal matrix: 2 on diagonal, -1 on off-diagonals
diag0 =  2.0 * t0 * np.ones(Np)
diagp = -1.0 * t0 * np.ones(Np-1)
H = np.diag(diag0) + np.diag(diagp, k=1) + np.diag(diagp, k=-1)

# -------------------------
# Eigen-decomposition
# -------------------------
E, V = np.linalg.eigh(H)   # E sorted ascending; columns of V are eigenvectors
order = np.argsort(E)
E = E[order]
V = V[:, order]

# -------------------------
# Analytic energies for infinite well of length L
# -------------------------
n = np.arange(1, Np+1)  # first Np modes
E_analytic = ((hbar*np.pi)**2 / (2.0*m*L**2) / q) * (n**2)

# -------------------------
# Wavefunction to show
# -------------------------
k = max(1, min(int(state_index), Np))  # clamp to valid range
psi_k = V[:, k-1]
# normalize (should already be, but for safety)
psi_k = psi_k / np.linalg.norm(psi_k)
P_k = np.abs(psi_k)**2

# X grid (interior points at i=1..Np)
X = a * np.arange(1, Np+1)

# -------------------------
# Figure 1: Energies (numeric vs analytic)
# -------------------------
plt.figure(figsize=(7, 5))
plt.plot(np.arange(1, Np+1), E, marker='o', linestyle='', label='Numeric (FD)')
plt.plot(np.arange(1, Np+1), E_analytic, linestyle='-', label='Analytic (infinite well)')
plt.xlabel("Eigenvalue index n")
plt.ylabel("Energy (eV)")
plt.title("Eigen-energies: Numeric vs Analytic")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# -------------------------
# Figure 2: Wavefunction / Probability for chosen state
# -------------------------
plt.figure(figsize=(7, 5))
plt.plot(X * 1e10, P_k, label=f"|ψ_{k}(x)|²")
plt.xlabel("Position x (Å)")
plt.ylabel("Probability density")
plt.title(f"Probability Density for State n = {k}")
plt.grid(True)
plt.tight_layout()
plt.show()

# Also print a small textual summary
print(f"Grid: Np = {Np}, a = {a:.2e} m, L ≈ {L*1e10:.2f} Å")
print(f"t0 = {t0:.6f} eV")
print(f"E_numeric[n={k}] = {E[k-1]:.6f} eV")
print(f"E_analytic[n={k}] = {E_analytic[k-1]:.6f} eV")
